# Chapter 6: Blocked GEMM Path

This notebook shows the first structural optimization in Chapter 6: tile reuse through blocked GEMM.

## What this notebook teaches

- How blocking changes the cost structure
- How to compare blocked GEMM against the naive baseline
- How to keep the benchmark record aligned with the chapter narrative

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "HANDOFF.md").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np

from examples.benchmark_record_template import set_benchmark_record
from examples.ch06_basic_ops_core import benchmark, gemm_blocked, gemm_naive, make_gemm_inputs

np.random.seed(0)
m = n = k = 64
bm, bn, bk = 32, 32, 16
a, b = make_gemm_inputs(m, n, k)
ref = a @ b
naive_out, naive_ms = benchmark(gemm_naive, a, b, runs=3)
out, elapsed_ms = benchmark(gemm_blocked, a, b, bm, bn, bk, runs=3)

print('Chapter 6: Blocked GEMM')
print('=' * 72)
print(f'Shape: M={m}, N={n}, K={k}')
print(f'Tile: bm={bm}, bn={bn}, bk={bk}')
print(f'Naive latency:   {naive_ms:.4f} ms per run')
print(f'Blocked latency:  {elapsed_ms:.4f} ms per run')
print(f'Max error: {np.max(np.abs(out - ref)):.2e}')
print()
print('Interpretation:')
print('- Blocking changes how data is reused inside a tile.')
print('- It is the bridge from correctness to structure-aware optimization.')
print()
record = set_benchmark_record(
    scenario='chapter 6 blocked gemm',
    operator='GEMM',
    platform='CPU',
    target='numpy',
    dtype='float32',
    shape=f'M={m},N={n},K={k}',
    baseline='naive GEMM',
    optimization=f'blocked GEMM (bm={bm},bn={bn},bk={bk})',
)
print('Benchmark record skeleton:')
for key in ['scenario', 'operator', 'platform', 'target', 'dtype', 'shape', 'baseline', 'optimization']:
    print(f'  {key}: {record[key]}')


## Takeaway

This notebook shows the first structural optimization in Chapter 6: the move from repeated global traffic to tile reuse.